# Experiment 2: Interview Text Feature Extraction & Assessment Pipeline
## Intelligent Multimodal Mock Interview Assessment System

### Objectives:
1. **Inspection of K-areem/AI-Interview-Questions Dataset**:
   - Validate that Parquet instruction pairs are expert reference targets, *not* graded candidate submissions.
2. **Text Normalization & Parsing**:
   - Parse raw `<s>[INST] ... [/INST] ... </s>` tokens into clean question and reference answer pairs.
3. **TF-IDF Semantic Space Construction**:
   - Fit unigram + bigram TF-IDF representations across the entire repository.
4. **Multi-Aspect Scoring Metrics**:
   - Semantic Content Relevance (Cosine similarity against reference answer).
   - Direct Question Alignment (Cosine similarity against prompt question).
   - Lexical Keyword Overlap (Jaccard similarity on technical content words).
   - Structural & Grammatical Coherence score.
5. **Custom Assessment Dataset Verification**:
   - Check `dataset/custom/interview_answer_assessment_template.csv` structure without generating synthetic/fake labels.

In [ ]:
import os, sys
import pandas as pd
import numpy as np

# Add src to path
sys.path.append(os.path.join("..", "src"))
from text_pipeline import parse_instruction_parquet, InterviewTextAssessor, clean_text

train_parquet = os.path.join("..", "dataset", "interview_questions", "train-00000-of-00001.parquet")
df_train = parse_instruction_parquet(train_parquet)
print(f"Parsed {len(df_train)} reference interview pairs.")
df_train.head(3)

## 2. Benchmark Scoring Demonstration across Quality Tiers

In [ ]:
assessor = InterviewTextAssessor()
corpus = list(df_train["question"]) + list(df_train["reference_answer"])
assessor.fit_corpus(corpus)

sample_q = df_train.iloc[0]["question"]
sample_ref = df_train.iloc[0]["reference_answer"]

candidate_responses = {
    "Strong": "UML facilitates specification in the analysis, design, and implementation by providing detailed diagrams and standard notations that capture both static structure and dynamic behavior.",
    "Moderate": "UML is used by software engineers to draw diagrams like class and use case diagrams for an app.",
    "Irrelevant": "I like writing software code on my personal laptop."
}

results = []
for tier, ans in candidate_responses.items():
    metrics = assessor.assess_answer(sample_q, ans, sample_ref)
    metrics["tier"] = tier
    results.append(metrics)

pd.DataFrame(results)